# Hypothesis Testing for Data Drift Monitoring: Catching Shift Without Crying Wolf

**Topic:** Statistics (hypothesis testing, multiple comparisons) — with an ML-quality-engineering angle (drift monitoring).

### Problem statement

You are the QA/quality engineer for a wine-chemistry classifier that was trained on
lab measurements from one cultivar's production batches (`load_wine`, class `1`). In
production, the pipeline occasionally receives wine samples whose chemistry comes from
a *different* cultivar or vintage mix, and your job is to build the **automated check**
that decides whether an incoming batch has drifted far enough from the training
distribution that the model's validity should be questioned — without flooding the
on-call channel with false alarms caused by ordinary sampling noise.

Chasing that one feature at a time at `alpha = 0.05` is the classic trap: test 13
features independently and you will get a "significant" result from pure chance more
often than you think. A real monitoring check has to account for that.

### What you should produce

1. `ks_test_per_feature(reference_df, current_df)` — runs a two-sample
   Kolmogorov–Smirnov test independently for every shared numeric column and returns a
   tidy DataFrame of `feature, ks_stat, p_value`.
2. `population_stability_index(reference_series, current_series, bins=10)` — computes
   the PSI for one feature, binning on the **reference** distribution's quantiles.
3. `correct_pvalues(pvalue_df, alpha=0.05)` — adds two columns of corrected
   significance flags to the KS-test table: `bonferroni_sig` and `bh_fdr_sig`
   (Benjamini–Hochberg). Implement both corrections yourself with `numpy`/`pandas`
   (no `statsmodels`).
4. `drift_report(reference_df, current_df, alpha=0.05)` — combines 1–3 into one table
   (per feature: `ks_stat`, `p_value`, `bonferroni_sig`, `bh_fdr_sig`, `psi`,
   `psi_band`) plus an overall verdict string, one of `"PASS"`, `"WARN"`, `"FAIL"`,
   using these rules:
   - `psi_band`: `"none"` if `psi < 0.1`, `"moderate"` if `0.1 <= psi < 0.25`, else
     `"major"`.
   - Overall verdict: `"FAIL"` if **any** feature is `bh_fdr_sig` **and** has
     `psi_band == "major"`; `"WARN"` if any feature is `bh_fdr_sig` (but none hit
     `"major"`) or any feature has `psi_band == "major"` without being significant;
     otherwise `"PASS"`.
5. Run `drift_report` on **two** reference/current pairs (built in the starter cells
   below) and write 3–4 sentences in a markdown cell comparing the two reports: which
   one should actually page someone, and — specifically for the feature(s) where the
   raw p-value and the corrected flags disagree — explain *why* the disagreement
   happens and why the corrected answer is the one you'd trust in production.


## Setup

Imports and the two reference/current batch pairs you'll run the report on.

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
from sklearn.datasets import load_wine

rng = np.random.default_rng(42)

wine = load_wine(as_frame=True)
df = wine.frame
feature_cols = [c for c in df.columns if c != "target"]

df[["target"] + feature_cols].groupby("target").size()

### Case A — real shift
`reference` = cultivar `1` (what the model was trained/validated on), `current` =
cultivar `2` (a genuinely different population arriving in production).

### Case B — no real shift
`reference` = a random half of cultivar `1`, `current` = the *other* random half of the
same cultivar. Same underlying distribution — any "significant" result here is a false
alarm from sampling noise, not real drift.

In [ ]:
# Case A: real drift between two different cultivars
reference_a = df[df["target"] == 1][feature_cols].reset_index(drop=True)
current_a = df[df["target"] == 2][feature_cols].reset_index(drop=True)

# Case B: no real drift -- random split of the SAME cultivar
same_cultivar = df[df["target"] == 1][feature_cols].reset_index(drop=True)
shuffled_idx = rng.permutation(len(same_cultivar))
half = len(shuffled_idx) // 2
reference_b = same_cultivar.iloc[shuffled_idx[:half]].reset_index(drop=True)
current_b = same_cultivar.iloc[shuffled_idx[half:]].reset_index(drop=True)

reference_a.shape, current_a.shape, reference_b.shape, current_b.shape

## TODO: implement the four functions described in the problem statement

In [ ]:
def ks_test_per_feature(reference_df: pd.DataFrame, current_df: pd.DataFrame) -> pd.DataFrame:
    '''Two-sample KS test for every shared numeric column.

    Returns a DataFrame with columns: feature, ks_stat, p_value.
    '''
    # TODO: implement
    raise NotImplementedError


def population_stability_index(reference_series: pd.Series, current_series: pd.Series, bins: int = 10) -> float:
    '''PSI between two 1-D samples, binned on the reference distribution's quantiles.'''
    # TODO: implement
    raise NotImplementedError


def correct_pvalues(pvalue_df: pd.DataFrame, alpha: float = 0.05) -> pd.DataFrame:
    '''Add 'bonferroni_sig' and 'bh_fdr_sig' boolean columns to pvalue_df.

    pvalue_df has at least a 'p_value' column (output of ks_test_per_feature).
    Implement Bonferroni and Benjamini-Hochberg yourself.
    '''
    # TODO: implement
    raise NotImplementedError


def drift_report(reference_df: pd.DataFrame, current_df: pd.DataFrame, alpha: float = 0.05):
    '''Build the full per-feature drift table plus an overall PASS/WARN/FAIL verdict.

    Returns (report_df, verdict_str). See the problem statement for the exact
    psi_band and verdict rules.
    '''
    # TODO: implement
    raise NotImplementedError


# TODO: call drift_report on (reference_a, current_a) and (reference_b, current_b),
# print both reports and verdicts, then write your comparison in a markdown cell below.

*(Write your 3–4 sentence comparison of Case A vs Case B here once you've run both reports.)*

---

## Solution

*(scroll down when you're ready — try it yourself first)*

<details>
<summary>Click to reveal solution</summary>

**Why multiple-testing correction matters here:** testing 13 features independently at
`alpha = 0.05` means that even when *nothing* has actually drifted (Case B), you expect
roughly `13 * 0.05 ≈ 0.65` false positives per run just from sampling noise — and
indeed, a naive run flags `nonflavanoid_phenols` (p ≈ 0.0044) as "significant" even
though both samples come from the same cultivar. Bonferroni (threshold `alpha/13 ≈
0.00385`) and Benjamini–Hochberg both correctly refuse to flag it once you account for
running 13 tests instead of one. In Case A, almost every feature is genuinely and
overwhelmingly shifted (cultivar 1 vs. cultivar 2 are chemically very different wines),
but `alcalinity_of_ash` is the interesting edge case: its raw p-value (~0.012) clears
the naive `alpha=0.05` bar, yet it is the only feature that **fails** the Bonferroni
correction (just barely) while still passing the less conservative BH-FDR correction —
exactly the kind of borderline case where the choice of correction method changes the
monitoring decision, and why a production drift check should default to the method the
team has agreed on (BH-FDR is the more common choice for monitoring, since Bonferroni's
conservatism trades missed real drift for fewer false alarms).

```python
def ks_test_per_feature(reference_df, current_df):
    rows = []
    for col in reference_df.columns:
        stat, p = stats.ks_2samp(reference_df[col], current_df[col])
        rows.append({"feature": col, "ks_stat": stat, "p_value": p})
    return pd.DataFrame(rows)


def population_stability_index(reference_series, current_series, bins=10):
    quantiles = np.linspace(0, 1, bins + 1)
    cut_points = np.unique(np.quantile(reference_series, quantiles))
    cut_points[0] = -np.inf
    cut_points[-1] = np.inf
    ref_counts, _ = np.histogram(reference_series, bins=cut_points)
    cur_counts, _ = np.histogram(current_series, bins=cut_points)
    # avoid log(0) / division by zero for empty bins
    ref_pct = np.where(ref_counts == 0, 1e-4, ref_counts / ref_counts.sum())
    cur_pct = np.where(cur_counts == 0, 1e-4, cur_counts / cur_counts.sum())
    return float(np.sum((cur_pct - ref_pct) * np.log(cur_pct / ref_pct)))


def correct_pvalues(pvalue_df, alpha=0.05):
    out = pvalue_df.copy()
    m = len(out)

    # Bonferroni: reject if p < alpha / m
    out["bonferroni_sig"] = out["p_value"] < (alpha / m)

    # Benjamini-Hochberg FDR
    sorted_df = out.sort_values("p_value").reset_index()
    ranks = np.arange(1, m + 1)
    bh_threshold = ranks / m * alpha
    passed = sorted_df["p_value"].values <= bh_threshold
    # largest k such that p_(k) <= (k/m)*alpha; everything up to and including k is significant
    if passed.any():
        k = np.max(np.where(passed)[0])
        sig_mask = np.zeros(m, dtype=bool)
        sig_mask[: k + 1] = True
    else:
        sig_mask = np.zeros(m, dtype=bool)
    sorted_df["bh_fdr_sig"] = sig_mask
    out = out.merge(
        sorted_df[["index", "bh_fdr_sig"]].set_index("index"),
        left_index=True, right_index=True,
    )
    return out


def psi_band(psi):
    if psi < 0.1:
        return "none"
    if psi < 0.25:
        return "moderate"
    return "major"


def drift_report(reference_df, current_df, alpha=0.05):
    ks_df = ks_test_per_feature(reference_df, current_df)
    report = correct_pvalues(ks_df, alpha=alpha)
    report["psi"] = [
        population_stability_index(reference_df[f], current_df[f])
        for f in report["feature"]
    ]
    report["psi_band"] = report["psi"].apply(psi_band)

    any_major_and_sig = ((report["psi_band"] == "major") & report["bh_fdr_sig"]).any()
    any_sig = report["bh_fdr_sig"].any()
    any_major = (report["psi_band"] == "major").any()

    if any_major_and_sig:
        verdict = "FAIL"
    elif any_sig or any_major:
        verdict = "WARN"
    else:
        verdict = "PASS"

    return report.sort_values("p_value").reset_index(drop=True), verdict


report_a, verdict_a = drift_report(reference_a, current_a)
report_b, verdict_b = drift_report(reference_b, current_b)

print("Case A (cultivar 1 vs cultivar 2) verdict:", verdict_a)
print(report_a)
print()
print("Case B (same cultivar, random split) verdict:", verdict_b)
print(report_b)
```

**Comparison:** Case A correctly comes back `FAIL` — every feature is BH-FDR-significant
*and* in the `"major"` PSI band, which is exactly what you'd want flagged, since the
production batch really is a different cultivar.

Case B is the more interesting result, and it does **not** come back `PASS`: it comes
back `WARN`. Running the actual numbers shows why. The corrected hypothesis test does
its job perfectly — `nonflavanoid_phenols` is the one feature that looks significant at
the naive `alpha=0.05` (raw p ≈ 0.0044), and *every other feature* has a large,
unremarkable p-value (several above 0.5, one above 0.9). Once you apply Bonferroni or
BH-FDR, **zero** features remain significant: the correction correctly recognizes that
one "hit" out of 13 tests is exactly what you'd expect from noise, not evidence of
drift. So the p-value side of the report is clean.

The PSI side is not: with only ~35 samples per side split across 10 quantile bins (just
3–4 points per bin), PSI is extremely noisy, and it reports a `"major"` shift for 9 of
the 13 features — including ones whose KS p-value is 0.97, 0.88, and 0.67. That is a
real and important failure mode in its own right: **PSI's standard `<0.1 / 0.1–0.25 /
>0.25` thresholds assume enough samples per bin to estimate stable proportions, and
production drift checks that run PSI on small daily batches will falsely scream
`"major"` drift from binning noise alone.** Because our `drift_report` verdict treats
*either* "significant after correction" *or* "PSI major" as worth a `WARN`, Case B
correctly lands as "go take a look" rather than silently `PASS`ing — but a quality
engineer reading the full table (not just the verdict string) can see that the
*corrected p-values*, not the raw PSI bands, are the trustworthy signal at this sample
size, and would reasonably decide not to page anyone. This is the real lesson: a single
summary metric is never enough — you need the full table, and you need to know *which*
of your signals degrades gracefully with small samples (the corrected hypothesis test)
and which one doesn't (PSI with too few points per bin).

</details>